# How do networks handle sequences?
**Topics:** RNNs · Vanishing Gradients Through Time · LSTM & GRU · Why Transformers Won

> 📌 **Read this chapter for the ideas, not the equations.** Recurrent networks are largely
> historical for new work — transformers (DL6) replaced them for essentially every sequence
> task. But two things here still matter: the **vanishing-gradient problem**, which motivates
> residual connections, gating, and normalization everywhere in modern deep learning; and the
> **gating pattern**, which reappears in SwiGLU feed-forward blocks and in state-space models.
>
> Interviews still ask "why did transformers replace LSTMs?" You need the reasoning, not the
> gate formulas.

## 1. Recurrent Neural Networks

### What it is
- Processes a sequence one element at a time, carrying a hidden state forward
- `h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)`
- The **same weights** are applied at every timestep (weight sharing across time)

### Key intuition
- Reading a sentence word by word; the hidden state is your memory of what came before
- Everything the network knows about the past must fit in that one fixed-size vector — this
  bottleneck is the root of every problem below

### Two structural limits
1. **Sequential by construction.** Step *t* cannot start until step *t−1* finishes, so you
   cannot parallelize across the time dimension. On modern accelerators this is the fatal one.
2. **Fixed-size memory.** A single hidden vector must summarize arbitrarily long history.

In [ ]:
import numpy as np
np.random.seed(0)

# A minimal RNN forward pass — enough to see the recurrence and the bottleneck.
def rnn_forward(X, W_x, W_h, b, h0=None):
    """X: (T, d_in). Returns hidden states (T, d_hidden)."""
    T = X.shape[0]
    h = np.zeros(W_h.shape[0]) if h0 is None else h0
    states = []
    for t in range(T):
        h = np.tanh(X[t] @ W_x + h @ W_h + b)   # <-- depends on h from t-1
        states.append(h.copy())
    return np.array(states)

T, D_IN, D_H = 12, 8, 16
X = np.random.randn(T, D_IN)
W_x = np.random.randn(D_IN, D_H) * 0.3
W_h = np.random.randn(D_H, D_H) * 0.3
b = np.zeros(D_H)

states = rnn_forward(X, W_x, W_h, b)
print(f"input {X.shape} -> hidden states {states.shape}")
print(f"Everything the model knows at step {T-1} is compressed into {D_H} numbers.")
print()
print("Note the loop: step t reads h from step t-1. That data dependency is why an")
print("RNN cannot be parallelized across time, and why a transformer — which computes")
print("all positions simultaneously — trains so much faster on the same hardware.")

## 2. Vanishing and Exploding Gradients Through Time

### What it is
Backpropagating through T timesteps multiplies by the recurrent Jacobian T times. If the
relevant factor is consistently below 1, gradients shrink exponentially and early timesteps
stop learning. Above 1, they explode to NaN.

### Key intuition
`gradient ≈ (factor)^T`. At T = 100, a factor of 0.9 gives ~2.7e-5 and a factor of 1.1 gives
~13,780. There is almost no stable middle.

### Why this matters far beyond RNNs
This is the same failure that motivates **residual connections** in ResNets and transformers,
**normalization layers**, careful **weight initialization**, and **gradient clipping**. The
general fix — give the gradient an additive path that doesn't get multiplied down — is exactly
what LSTM's cell state does, and exactly what a skip connection does.

In [ ]:
# The exponential is the whole story.
print(f"{'steps':>7}{'factor 0.9':>16}{'factor 1.0':>13}{'factor 1.1':>16}")
print("-" * 54)
for T in [10, 25, 50, 100, 200]:
    print(f"{T:>7}{0.9**T:>16.2e}{1.0**T:>13.2f}{1.1**T:>16.2e}")

print()
print("Only factor == 1.0 is stable, and nothing keeps it there by default.")
print()
print("Now the fix that LSTM introduced, in one line:")
print("  multiplicative path:  grad *= f  at every step   -> exponential decay")
print("  additive path:        c_t = f*c_{t-1} + i*g      -> gradient flows when f ~ 1")
print()
print("A residual connection (x + F(x)) is the same trick, applied across depth")
print("instead of across time. Recognizing them as the same idea is the point of")
print("this chapter.")

## 3. LSTM and GRU: Gating

Both solve the vanishing-gradient problem the same way — a path the gradient can travel
without being repeatedly multiplied down — and differ only in how much machinery they use.

**LSTM** keeps two states: a cell state `c` (long-term, the gradient highway) and a hidden
state `h` (working memory). Four gates control it:

| Gate | Role |
|---|---|
| Forget `f` | How much of the cell state to keep |
| Input `i` | How much new candidate information to write |
| Candidate `g` | The new information itself |
| Output `o` | How much of the cell state to expose as `h` |

The update `c_t = f ⊙ c_{t-1} + i ⊙ g` is the important line. When `f ≈ 1` the cell state
passes through nearly unchanged and so does its gradient.

**GRU** merges forget and input into one update gate and drops the separate cell state — two
gates instead of four, ~25% fewer parameters, similar accuracy in practice.

**Choosing between them:** try GRU first (fewer parameters, faster, less prone to overfitting
on small data); switch to LSTM if you need finer memory control. In practice the difference is
usually smaller than the difference made by tuning anything else.

In [ ]:
# One LSTM step, to make the gating concrete.
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def lstm_step(x, h_prev, c_prev, W, U, b):
    """W: (d_in, 4*d_h), U: (d_h, 4*d_h). Gates stacked as [i, f, g, o]."""
    d_h = h_prev.shape[0]
    z = x @ W + h_prev @ U + b
    i, f, g, o = np.split(z, 4)
    i, f, o = sigmoid(i), sigmoid(f), sigmoid(o)
    g = np.tanh(g)
    c = f * c_prev + i * g          # <-- ADDITIVE: the gradient highway
    h = o * np.tanh(c)
    return h, c, {"forget": f.mean(), "input": i.mean(), "output": o.mean()}

D_IN, D_H = 8, 16
W = np.random.randn(D_IN, 4 * D_H) * 0.1
U = np.random.randn(D_H, 4 * D_H) * 0.1
b = np.zeros(4 * D_H)
b[D_H:2 * D_H] = 1.0            # forget-gate bias init to 1: "remember by default"

h, c = np.zeros(D_H), np.zeros(D_H)
print(f"{'step':>5}{'forget':>10}{'input':>9}{'output':>9}{'|c| mean':>11}")
print("-" * 45)
for t in range(6):
    h, c, gates = lstm_step(np.random.randn(D_IN), h, c, W, U, b)
    print(f"{t:>5}{gates['forget']:>10.3f}{gates['input']:>9.3f}"
          f"{gates['output']:>9.3f}{np.abs(c).mean():>11.4f}")

print()
print("Forget gate sits near 0.73 because we initialized its bias to 1.0. That is a real")
print("trick, not a toy detail: starting near 'remember everything' keeps the gradient")
print("path open early in training, when the gates have not learned anything yet.")

# Parameter comparison
lstm_p = 4 * (D_IN * D_H + D_H * D_H + D_H)
gru_p = 3 * (D_IN * D_H + D_H * D_H + D_H)
print(f"\nParameters at d_in={D_IN}, d_h={D_H}:  LSTM {lstm_p:,}   GRU {gru_p:,}"
      f"   ({1 - gru_p/lstm_p:.0%} fewer)")

## 4. Why Transformers Won

Gating fixed the gradient problem. It did not fix the other one.

| | RNN / LSTM / GRU | Transformer |
|---|---|---|
| **Training parallelism** | None across time — step t waits for t−1 | Full — all positions at once |
| **Path between two positions** | O(distance) steps | O(1) — direct attention |
| **Memory of the past** | One fixed-size vector | Every position, directly addressable |
| **Cost per layer** | O(T · d²) | O(T² · d) |
| **Scaling behaviour** | Plateaus | Keeps improving with data and parameters |

Note the cost row: **the transformer is asymptotically worse in sequence length.** It won
anyway, because O(T²) work you can do in parallel beats O(T) work you must do serially on
hardware with tens of thousands of cores. That trade — accept more total work in exchange for
parallelism — is one of the most important lessons in modern ML systems, and it generalizes
well beyond this example.

### Where recurrence still appears
- **Very long sequences on constrained hardware**, where O(T²) attention is simply unaffordable
- **Streaming / online inference** with strict constant-memory requirements
- **State-space models** (Mamba and relatives) — a modern revival of the recurrent idea with
  parallelizable training, explicitly designed to get both properties at once

### Interview Q&A

**Why did transformers replace LSTMs?**
- Not accuracy first — *parallelism*. An RNN's step t depends on step t−1, so training cannot
  use the width of a modern accelerator; a transformer computes all positions simultaneously
- Any two positions are one attention hop apart rather than O(distance) recurrent steps, so
  long-range dependencies are learned directly
- Because they train efficiently, transformers scale with data and parameters where RNNs plateaued

**How does an LSTM solve the vanishing gradient problem?**
- The cell state updates *additively*: `c_t = f ⊙ c_{t-1} + i ⊙ g`
- When the forget gate is near 1, gradient flows backward through that path nearly unchanged
  instead of being multiplied down at every step
- Same principle as a residual connection, applied across time instead of across depth

**When would you still choose a recurrent model?**
- Extremely long sequences where O(T²) attention is unaffordable
- Streaming inference needing constant memory regardless of history length
- Otherwise: a transformer, or a state-space model if you need both long context and recurrence

### Gotchas
- Bidirectional models see the whole sequence, so they're unusable for causal or streaming tasks
- Initialize the forget-gate bias to 1.0 — a small change with a real effect on early training
- `pack_padded_sequence` in PyTorch avoids computing over padding; forgetting it corrupts the
  final hidden state with padding tokens

## Key Takeaways
- RNNs carry a fixed-size hidden state; their fatal limit is the sequential dependency, not accuracy
- Vanishing/exploding gradients come from multiplying the recurrent Jacobian T times — `factor^T` has no stable middle
- LSTM/GRU gating fixes it with an *additive* path; residual connections are the same idea across depth
- GRU has 2 gates to LSTM's 4 (~25% fewer parameters) with comparable accuracy — try it first
- Transformers won on parallelism despite being asymptotically worse in sequence length
- "More total work, but parallel, beats less work done serially" is the general lesson worth carrying forward
- Recurrence survives in state-space models, which aim for parallel training *and* constant-memory inference